# FIT5196 Assessment 2 Task 1 Clean Tutorial Notebook

这个 notebook 是 Task 1 的干净教学版。设计原则是：

- notebook 展示每个 attribute 的具体 pandas / graph / model 代码；
- `task1_clean_tools.py` 只做通用壳子：记录、打印、before/after、validation table、export；
- 每一步都能看到 `flagged`, `fixed`, `remaining`, `skipped_by_tracker`。

## How to use fold / unfold blocks

这个 notebook 使用 Jupyter Markdown 支持的 HTML `<details>` 区块来做 fold / unfold。课堂讲解时可以先只看主线；需要解释依据、business rules 或细节时，再展开对应区块。

## 0. Assignment-given constraints first

<details open>
<summary><strong>展开 / 收起：assignment PDF 已经给出的规则</strong></summary>

在正式 EDA 之前，先把 assignment PDF 已经给出的规则翻译成 data constraints。这样后面的检查不是“碰运气找异常”，而是验证哪些 rows 违反了已知规则。

| Rule from assignment | How we use it from the start |
| --- | --- |
| Dirty data 每行最多一个 anomaly | 使用 issue tracker；每行只接受一个最终修复原因 |
| Dirty data 的 `order_id`, `time`, `delivery_fee`, item quantities 是 error-free | 不修改这些字段；用 `order_id` 推 `branch_code`，用 `time` 推 `order_type` |
| Missing data 只有 coverage/missing anomalies | 非缺失字段可作为干净参考；用 missing 文件内部非缺失 `delivery_fee` 训练模型 |
| Outlier data 只有 `delivery_fee` outliers | 不修其他字段；只检测并删除 delivery fee outlier rows |
| `customer_lat/customer_lon` 来自 `nodes.csv` | 坐标应能 exact match 到 node；优先检查 sign flip 和 lat/lon swap |
| `distance_to_customer_KM` 是 Dijkstra shortest path | 使用 `edges.csv` 的 `distance(m)` 作为 graph weight，不用 straight-line distance |
| `delivery_fee` branch-specific 且线性依赖 weekend/time/distance | 每个 branch 单独训练 linear model，并加入 `is_weekend`, `time_code`, `distance_to_customer_KM` |
| Loyalty customer 有 50% delivery fee discount | 建模前还原 undiscounted fee，预测后再应用折扣 |

所以这个 clean notebook 的顺序是：先声明规则，再做 EDA，再按 dependency 顺序修复和验证。

</details>

## Attribute dependency map

<details open>
<summary><strong>展开 / 收起：attribute dependency map</strong></summary>

```mermaid
flowchart LR
    order_id[order_id] --> branch_code[branch_code]

    date[date] --> parsed_date[parsed date]
    parsed_date --> is_weekend[is_weekend]

    time[time] --> order_type[order_type]
    order_type --> menu[meal menu]
    order_items[order_items] --> menu
    menu --> order_price[order_price]

    branch_code --> branch_node[branch node]
    customer_lat[customer_lat] --> customer_node[customer node]
    customer_lon[customer_lon] --> customer_node
    branch_node --> road_graph[road graph]
    customer_node --> road_graph
    road_graph --> distance[distance_to_customer_KM]

    is_weekend --> delivery_fee[delivery_fee]
    order_type --> delivery_fee
    distance --> delivery_fee
    branch_code --> delivery_fee
    loyalty[customerHasloyalty?] --> delivery_fee
```

</details>

## 1. Setup

这里只导入通用 helper 和需要展示给学生看的核心库。helper 不包含 attribute-specific cleaning 逻辑。

In [30]:
from pathlib import Path
from datetime import datetime
import ast

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

from task1_clean_tools import Task1NotebookTools

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

DATA_DIR = Path(".")
task = Task1NotebookTools(DATA_DIR)


## 1.1 How the helper works

`Task1NotebookTools` 是一个很薄的 helper。它不决定哪个 attribute 错，也不决定应该怎么修；这些判断都写在 notebook 里。

它主要帮我们统一做五件事：

1. 保存原始数据和 working copy；
2. 对 dirty data 维护 one-anomaly tracker；
3. 根据我们传入的 `issue_mask` 和 `replacements` 应用修复；
4. 打印 `flagged / fixed / remaining / skipped_by_tracker`；
5. 展示 before/after sample，最后导出 CSV。

核心调用长这样：

```python
task.fix_values(
    dataset="dirty",
    step="branch_code",
    issue_mask=branch_issue,
    replacements={"branch_code": expected_branch},
    sample_columns=["branch_code"],
    remaining_check=lambda df: ...,
)
```

参数含义：

| Parameter | Meaning |
| --- | --- |
| `dataset` | 当前处理哪个数据集：`dirty`, `missing`, `outlier` |
| `step` | 当前修哪个 attribute，也会写进 tracker/log |
| `issue_mask` | 一个 True/False Series，表示哪些 rows 被怀疑有这个问题 |
| `replacements` | 一个 dict，key 是要改的 column，value 是 replacement Series 或固定值 |
| `sample_columns` | before/after 展示哪些 columns |
| `remaining_check` | 修完后重新检查还有多少同类问题 |

对 dirty data，helper 会额外做一件事：如果某行已经被前面某一步修过，它不会再被后面的步骤改动，而是计入 `skipped_by_tracker`。这正好对应 assignment rule：dirty data 每行最多一个 anomaly。


In [31]:
# Mini example: how a boolean mask and replacement Series work
example = pd.DataFrame({
    "order_id": ["ORDA00001", "ORDJ00002"],
    "branch_code": ["tp", "TP"],
})

example_prefix_map = {"A": "BK", "J": "TP"}
example_expected = example["order_id"].str[3].map(example_prefix_map)
example_issue_mask = example["branch_code"].astype(str).ne(example_expected)

example_demo = example.copy()
example_demo.loc[example_issue_mask, "branch_code"] = example_expected[example_issue_mask]

display(example)
print("issue_mask:")
display(example_issue_mask)
print("after replacement:")
display(example_demo)

,order_id,branch_code
0,ORDA00001,tp
1,ORDJ00002,TP


issue_mask:


0     True
1    False
dtype: bool

after replacement:


,order_id,branch_code
0,ORDA00001,BK
1,ORDJ00002,TP



## 1.2 If you build this helper yourself

如果学生自己做 assignment，不需要一开始就写 class。更自然的顺序是：

1. 先用一个小 DataFrame 手写 `issue_mask`；
2. 再手写 `.loc[mask, column] = replacement`；
3. 发现每个 attribute 都要重复 before/after、计数、remaining check；
4. 最后才把重复部分包装成 helper function。

也就是说，helper 不是为了隐藏逻辑，而是为了避免每一步都重复写同样的 reporting code。

下面这个 mini helper 就是 `fix_values()` 的简化版。它只做三件事：

- 用 mask 找出要修的 rows；
- 应用 replacement；
- 打印 fixed 和 remaining。


In [32]:
student_example = pd.DataFrame({
    "order_id": ["ORDA00001", "ORDJ00002", "ORDK00003"],
    "branch_code": ["tp", "TP", "BK"],
})

student_prefix_map = {"A": "BK", "J": "TP", "K": "BK"}
student_expected = student_example["order_id"].str[3].map(student_prefix_map)
student_issue = student_example["branch_code"].astype(str).ne(student_expected)

before = student_example.loc[student_issue, ["order_id", "branch_code"]].copy()
student_example.loc[student_issue, "branch_code"] = student_expected.loc[student_issue]
after = student_example.loc[student_issue, ["order_id", "branch_code"]].copy()

remaining = student_example["branch_code"].ne(student_example["order_id"].str[3].map(student_prefix_map)).sum()

print(f"fixed={int(student_issue.sum())}, remaining={int(remaining)}")
pd.concat([before.add_suffix("_before"), after.add_suffix("_after")], axis=1)


fixed=1, remaining=0


,order_id_before,branch_code_before,order_id_after,branch_code_after
0,ORDA00001,tp,ORDA00001,BK



这个 mini helper 还没有处理 dirty data 的 one-anomaly tracker，也没有处理多列 replacement，比如 customer coordinates 的 lat/lon 同时变化。

我们的 `Task1NotebookTools.fix_values()` 只是把这个 mini version 扩展成 assignment 需要的版本：

- 支持多个 columns 一起改；
- 支持 dirty tracker，避免一行被重复修；
- 支持统一的 before/after sample；
- 支持每一步写入 log，最后能生成 `step_log()`。

课堂上可以先理解 mini version，再看正式 helper 的参数。


## 2. Overview

先看三个文件的基本结构。注意：schema 一样不代表处理方法一样，因为 assignment 已经定义了三类不同问题。

In [33]:
task.overview()

Loaded Task 1 files.


,dataset,rows,columns,missing_cells,duplicated_order_id
0,dirty,500,12,0,0
1,missing,500,12,200,0
2,outlier,500,12,0,0


In [34]:
task.column_summary("dirty")

[dirty] rows=500, columns=12


,column,dtype,missing,unique
0,order_id,str,0,500
1,date,str,0,295
2,time,str,0,72
3,order_type,str,0,3
4,branch_code,str,0,6
5,order_items,str,0,499
6,order_price,float64,0,428
7,customer_lat,float64,0,493
8,customer_lon,float64,0,493
9,customerHasloyalty?,int64,0,2


## 3. Business rules as code

这一格很重要：我们把 assignment PDF 的文字规则翻译成代码。后面每个 attribute 的检查都会直接使用这些 rule functions。

In [35]:
PREFIX_TO_BRANCH = {
    **dict.fromkeys(list("AKX"), "BK"),
    **dict.fromkeys(list("BJY"), "TP"),
    **dict.fromkeys(list("CIZ"), "NS"),
}

MENU = {
    "Breakfast": {"Cereal": 21.00, "Coffee": 7.50, "Eggs": 22.00, "Pancake": 24.25},
    "Lunch": {"Burger": 31.00, "Chicken": 32.00, "Fries": 12.00, "Salad": 17.20, "Steak": 45.00},
    "Dinner": {"Fish&Chips": 35.00, "Pasta": 27.50, "Salmon": 41.00, "Shrimp": 54.00},
}

TIME_CODE = {"Breakfast": 0, "Lunch": 1, "Dinner": 2}


def branch_from_order_id(order_id):
    return PREFIX_TO_BRANCH[str(order_id)[3]]


def parse_items(value):
    return ast.literal_eval(value)


def format_items(items):
    return str([(str(item), int(quantity)) for item, quantity in items])


def item_total(items, order_type):
    return round(sum(MENU[order_type][item] * quantity for item, quantity in items), 2)


def expected_order_price(row):
    return item_total(parse_items(row["order_items"]), row["order_type"])


## 4. Road graph as code

`distance_to_customer_KM` 要用 road network shortest path。这里展示如何从 `nodes.csv`, `branches.csv`, `edges.csv` 建 graph。

In [36]:
road_graph = nx.Graph()
for _, row in task.edges.iterrows():
    road_graph.add_edge(
        int(row["u"]),
        int(row["v"]),
        weight=float(row["distance(m)"]),
    )

coord_to_node = {
    (round(float(row["lat"]), 7), round(float(row["lon"]), 7)): int(row["node"])
    for _, row in task.nodes.iterrows()
}

branch_to_node = {}
for _, row in task.branches.iterrows():
    key = (round(float(row["branch_lat"]), 7), round(float(row["branch_lon"]), 7))
    branch_to_node[row["branch_code"]] = coord_to_node[key]

distance_cache = {}


def node_for_customer(lat, lon):
    key = (round(float(lat), 7), round(float(lon), 7))
    return coord_to_node[key]


def expected_distance(row):
    branch_node = branch_to_node[row["branch_code"]]
    customer_node = node_for_customer(row["customer_lat"], row["customer_lon"])
    cache_key = (branch_node, customer_node)
    if cache_key not in distance_cache:
        metres = nx.shortest_path_length(
            road_graph,
            branch_node,
            customer_node,
            weight="weight",
        )
        distance_cache[cache_key] = round(metres / 1000, 3)
    return distance_cache[cache_key]

print("graph nodes:", road_graph.number_of_nodes())
print("graph edges:", road_graph.number_of_edges())
print("branch nodes:", branch_to_node)

graph nodes: 17117
graph edges: 25491
branch nodes: {'NS': 2455254505, 'TP': 1390575046, 'BK': 1889485053}


## 5. Dirty data

现在开始 dirty data。每一步 notebook 负责算 `issue_mask` 和 replacement，helper 负责应用、记录、展示。

In [37]:
dirty = task.start("dirty")

[dirty:start] protected columns: order_id, time, delivery_fee, item quantities


### 5.1 branch_code

`order_id` prefix 决定 expected branch。这里同时覆盖大小写错误和 prefix mismatch。

In [38]:
expected_branch = dirty["order_id"].map(branch_from_order_id)
branch_issue = dirty["branch_code"].astype(str).ne(expected_branch)

task.fix_values(
    dataset="dirty",
    step="branch_code",
    issue_mask=branch_issue,
    replacements={"branch_code": expected_branch},
    sample_columns=["branch_code"],
    remaining_check=lambda df: df["branch_code"].astype(str).ne(df["order_id"].map(branch_from_order_id)),
)

[dirty:branch_code] flagged=37, fixed=37, remaining=0, skipped_by_tracker=0


,order_id_before,branch_code_before,order_id_after,branch_code_after
4,ORDK03173,tp,ORDK03173,BK
21,ORDI09297,BK,ORDI09297,NS
22,ORDC07228,TP,ORDC07228,NS
31,ORDK06897,ns,ORDK06897,BK
33,ORDA05281,tp,ORDA05281,BK
48,ORDI09968,tp,ORDI09968,NS
52,ORDK07377,ns,ORDK07377,BK
74,ORDK04119,NS,ORDK04119,BK


### 5.2 date

把多种可解释的日期格式统一成 `YYYY-MM-DD`。

In [39]:
DATE_FORMATS = ("%Y-%m-%d", "%Y-%d-%m", "%m-%d-%Y")


def clean_date(value):
    for date_format in DATE_FORMATS:
        parsed = pd.to_datetime(value, format=date_format, errors="coerce")
        if pd.notna(parsed) and parsed.year == 2018:
            return parsed.strftime("%Y-%m-%d")
    return pd.NaT


In [40]:
canonical_date = dirty["date"].map(clean_date)
date_issue = canonical_date.ne(dirty["date"])

task.fix_values(
    dataset="dirty",
    step="date",
    issue_mask=date_issue,
    replacements={"date": canonical_date},
    sample_columns=["date"],
    remaining_check=lambda df: df["date"].map(clean_date).ne(df["date"]),
)

[dirty:date] flagged=37, fixed=37, remaining=0, skipped_by_tracker=0


,order_id_before,date_before,order_id_after,date_after
0,ORDX00699,03-08-2018,ORDX00699,2018-03-08
10,ORDC01147,2018-26-08,ORDC01147,2018-08-26
16,ORDK04564,2018-19-09,ORDK04564,2018-09-19
18,ORDY05205,04-01-2018,ORDY05205,2018-04-01
37,ORDJ05383,2018-18-08,ORDJ05383,2018-08-18
42,ORDJ08299,02-09-2018,ORDJ08299,2018-02-09
51,ORDB00774,2018-16-06,ORDB00774,2018-06-16
70,ORDB07017,04-07-2018,ORDB07017,2018-04-07


### 5.3 order_type

`time` 是 protected，所以用 meal window 反推 `order_type`。

In [41]:
def order_type_from_time(value):
    hour, minute, second = map(int, str(value).split(":"))
    total_seconds = hour * 3600 + minute * 60 + second

    if 8 * 3600 <= total_seconds <= 12 * 3600:
        return "Breakfast"
    if 12 * 3600 < total_seconds <= 16 * 3600:
        return "Lunch"
    if 16 * 3600 < total_seconds <= 20 * 3600:
        return "Dinner"
    return np.nan


In [42]:
expected_type = dirty["time"].map(order_type_from_time)
order_type_issue = dirty["order_type"].ne(expected_type)

task.fix_values(
    dataset="dirty",
    step="order_type",
    issue_mask=order_type_issue,
    replacements={"order_type": expected_type},
    sample_columns=["time", "order_type"],
    remaining_check=lambda df: df["order_type"].ne(df["time"].map(order_type_from_time)),
)

[dirty:order_type] flagged=37, fixed=37, remaining=0, skipped_by_tracker=0


,order_id_before,time_before,order_type_before,order_id_after,time_after,order_type_after
23,ORDX10183,18:28:43,Lunch,ORDX10183,18:28:43,Dinner
28,ORDY07273,13:34:38,Breakfast,ORDY07273,13:34:38,Lunch
43,ORDK02724,13:24:30,Breakfast,ORDK02724,13:24:30,Lunch
45,ORDI00689,12:33:48,Breakfast,ORDI00689,12:33:48,Lunch
67,ORDJ03210,08:20:16,Lunch,ORDJ03210,08:20:16,Breakfast
69,ORDJ10027,12:23:39,Breakfast,ORDJ10027,12:23:39,Lunch
95,ORDK04641,10:01:41,Lunch,ORDK04641,10:01:41,Breakfast
108,ORDJ06140,12:03:22,Breakfast,ORDJ06140,12:03:22,Lunch


### 5.4 customer coordinates

Melbourne customers only，且 customer coordinates 来自 `nodes.csv`。明显问题是 latitude 正负号错误，或 lat/lon 被交换。

In [43]:
swap_issue = dirty["customer_lat"].gt(90) & dirty["customer_lon"].lt(0)
sign_issue = dirty["customer_lat"].between(0, 90)
coordinate_issue = swap_issue | sign_issue

new_lat = dirty["customer_lat"].copy()
new_lon = dirty["customer_lon"].copy()

new_lat.loc[swap_issue] = dirty.loc[swap_issue, "customer_lon"]
new_lon.loc[swap_issue] = dirty.loc[swap_issue, "customer_lat"]
new_lat.loc[sign_issue] = -dirty.loc[sign_issue, "customer_lat"]

task.fix_values(
    dataset="dirty",
    step="customer_coordinates",
    issue_mask=coordinate_issue,
    replacements={"customer_lat": new_lat, "customer_lon": new_lon},
    sample_columns=["customer_lat", "customer_lon"],
    remaining_check=lambda df: df["customer_lat"].gt(0) | df["customer_lon"].lt(0),
)

[dirty:customer_coordinates] flagged=41, fixed=41, remaining=0, skipped_by_tracker=0


,order_id_before,customer_lat_before,customer_lon_before,order_id_after,customer_lat_after,customer_lon_after
32,ORDZ03318,37.806231,144.939527,ORDZ03318,-37.806231,144.939527
38,ORDZ06323,37.814201,144.960980,ORDZ06323,-37.814201,144.960980
57,ORDK02131,37.805558,144.948692,ORDK02131,-37.805558,144.948692
62,ORDX01429,37.814187,144.950255,ORDX01429,-37.814187,144.950255
66,ORDA02101,37.822164,145.004077,ORDA02101,-37.822164,145.004077
78,ORDB06092,37.808330,144.958472,ORDB06092,-37.808330,144.958472
79,ORDB00776,37.810334,144.947065,ORDB00776,-37.810334,144.947065
93,ORDZ02223,37.824301,144.954420,ORDZ02223,-37.824301,144.954420


### 5.5 order_items

数量是 protected，不改 quantity。若 item 不属于当前 meal menu，就用 `order_price` 反推出唯一 item name。

In [44]:
def repair_order_items(row):
    items = parse_items(row["order_items"])
    valid_items = set(MENU[row["order_type"]])
    wrong_positions = [i for i, (item, _) in enumerate(items) if item not in valid_items]

    if not wrong_positions:
        return row["order_items"]

    target_price = round(float(row["order_price"]), 2)
    for position in wrong_positions:
        quantity = items[position][1]
        for candidate in MENU[row["order_type"]]:
            candidate_items = list(items)
            candidate_items[position] = (candidate, quantity)
            if abs(item_total(candidate_items, row["order_type"]) - target_price) < 0.01:
                return format_items(candidate_items)

    return row["order_items"]


repaired_items = dirty.apply(repair_order_items, axis=1)
items_issue = repaired_items.ne(dirty["order_items"])

task.fix_values(
    dataset="dirty",
    step="order_items",
    issue_mask=items_issue,
    replacements={"order_items": repaired_items},
    sample_columns=["order_type", "order_items", "order_price"],
    remaining_check=lambda df: df.apply(repair_order_items, axis=1).ne(df["order_items"]),
)

[dirty:order_items] flagged=37, fixed=37, remaining=0, skipped_by_tracker=0


,order_id_before,order_type_before,order_items_before,order_price_before,order_id_after,order_type_after,order_items_after,order_price_after
15,ORDB07869,Lunch,"[('Salad', 10), ('Fries', 10), ('Chicken', 6),...",912.0,ORDB07869,Lunch,"[('Salad', 10), ('Fries', 10), ('Chicken', 6),...",912.0
29,ORDX02818,Lunch,"[('Fries', 3), ('Salad', 1), ('Fish&Chips', 8)]",301.2,ORDX02818,Lunch,"[('Fries', 3), ('Salad', 1), ('Burger', 8)]",301.2
80,ORDC01608,Lunch,"[('Salad', 8), ('Fries', 5), ('Burger', 7), ('...",792.6,ORDC01608,Lunch,"[('Salad', 8), ('Fries', 5), ('Burger', 7), ('...",792.6
106,ORDK07696,Lunch,"[('Shrimp', 9), ('Steak', 7), ('Salad', 8)]",560.6,ORDK07696,Lunch,"[('Fries', 9), ('Steak', 7), ('Salad', 8)]",560.6
130,ORDZ10295,Lunch,"[('Burger', 4), ('Eggs', 6)]",394.0,ORDZ10295,Lunch,"[('Burger', 4), ('Steak', 6)]",394.0
135,ORDY02309,Breakfast,"[('Pancake', 6), ('Cereal', 7), ('Salmon', 2),...",351.5,ORDY02309,Breakfast,"[('Pancake', 6), ('Cereal', 7), ('Eggs', 2), (...",351.5
143,ORDJ08271,Lunch,"[('Fries', 5), ('Pasta', 1)]",91.0,ORDJ08271,Lunch,"[('Fries', 5), ('Burger', 1)]",91.0
147,ORDB01016,Dinner,"[('Fries', 9), ('Pasta', 5)]",452.5,ORDB01016,Dinner,"[('Fish&Chips', 9), ('Pasta', 5)]",452.5


### 5.6 order_price

items 合法后，`order_price` 应该等于菜单单价乘数量的总和。

In [45]:
expected_price = dirty.apply(expected_order_price, axis=1)
price_issue = dirty["order_price"].sub(expected_price).abs().gt(0.01)

task.fix_values(
    dataset="dirty",
    step="order_price",
    issue_mask=price_issue,
    replacements={"order_price": expected_price},
    sample_columns=["order_items", "order_price"],
    remaining_check=lambda df: df["order_price"].sub(df.apply(expected_order_price, axis=1)).abs().gt(0.01),
)

[dirty:order_price] flagged=37, fixed=37, remaining=0, skipped_by_tracker=0


,order_id_before,order_items_before,order_price_before,order_id_after,order_items_after,order_price_after
24,ORDA10507,"[('Eggs', 10), ('Coffee', 6), ('Pancake', 10)]",173.0,ORDA10507,"[('Eggs', 10), ('Coffee', 6), ('Pancake', 10)]",507.50
50,ORDC08998,"[('Eggs', 3), ('Pancake', 5), ('Coffee', 5), (...",559.6,ORDC08998,"[('Eggs', 3), ('Pancake', 5), ('Coffee', 5), (...",266.75
59,ORDJ06050,"[('Pancake', 10), ('Cereal', 7), ('Coffee', 9)]",1239.5,ORDJ06050,"[('Pancake', 10), ('Cereal', 7), ('Coffee', 9)]",457.00
64,ORDA07051,"[('Eggs', 9), ('Cereal', 9)]",967.0,ORDA07051,"[('Eggs', 9), ('Cereal', 9)]",387.00
71,ORDA09458,"[('Pasta', 10), ('Fish&Chips', 8), ('Salmon', 6)]",907.5,ORDA09458,"[('Pasta', 10), ('Fish&Chips', 8), ('Salmon', 6)]",801.00
73,ORDI09699,"[('Pancake', 4), ('Eggs', 8)]",43.0,ORDI09699,"[('Pancake', 4), ('Eggs', 8)]",273.00
84,ORDZ09320,"[('Eggs', 3), ('Cereal', 3), ('Coffee', 10), (...",367.0,ORDZ09320,"[('Eggs', 3), ('Cereal', 3), ('Coffee', 10), (...",252.50
88,ORDY08024,"[('Pancake', 9), ('Cereal', 8), ('Coffee', 6)]",145.5,ORDY08024,"[('Pancake', 9), ('Cereal', 8), ('Coffee', 6)]",431.25


### 5.7 distance_to_customer_KM

最后检查 graph distance。这里依赖已经修好的 branch 和 customer coordinates。

In [46]:
expected_distance_dirty = dirty.apply(expected_distance, axis=1)
distance_issue = dirty["distance_to_customer_KM"].sub(expected_distance_dirty).abs().gt(0.001)

task.fix_values(
    dataset="dirty",
    step="distance_to_customer_KM",
    issue_mask=distance_issue,
    replacements={"distance_to_customer_KM": expected_distance_dirty},
    sample_columns=["branch_code", "customer_lat", "customer_lon", "distance_to_customer_KM"],
    remaining_check=lambda df: df["distance_to_customer_KM"].sub(df.apply(expected_distance, axis=1)).abs().gt(0.001),
)

[dirty:distance_to_customer_KM] flagged=37, fixed=37, remaining=0, skipped_by_tracker=0


,order_id_before,branch_code_before,customer_lat_before,customer_lon_before,distance_to_customer_KM_before,order_id_after,branch_code_after,customer_lat_after,customer_lon_after,distance_to_customer_KM_after
8,ORDY07813,TP,-37.800007,144.935683,7.651,ORDY07813,TP,-37.800007,144.935683,12.847
12,ORDJ08517,TP,-37.800506,144.964703,7.962,ORDJ08517,TP,-37.800506,144.964703,9.764
19,ORDZ08993,NS,-37.813351,144.965110,7.066,ORDZ08993,NS,-37.813351,144.965110,7.857
25,ORDY03212,TP,-37.805766,144.955083,7.975,ORDY03212,TP,-37.805766,144.955083,9.369
58,ORDJ10405,TP,-37.799046,144.978352,11.323,ORDJ10405,TP,-37.799046,144.978352,10.327
91,ORDC00387,NS,-37.801997,144.963880,9.979,ORDC00387,NS,-37.801997,144.963880,6.860
110,ORDC03440,NS,-37.798845,144.953774,4.975,ORDC03440,NS,-37.798845,144.953774,7.651
152,ORDX09640,BK,-37.817079,145.009257,8.562,ORDX09640,BK,-37.817079,145.009257,4.011


### 5.8 dirty validation

Final validation 只验证，不在这里偷偷改。

In [47]:
expected_branch_final = dirty["order_id"].map(branch_from_order_id)
expected_type_final = dirty["time"].map(order_type_from_time)
expected_price_final = dirty.apply(expected_order_price, axis=1)
expected_distance_final = dirty.apply(expected_distance, axis=1)
protected_columns = ["order_id", "time", "delivery_fee"]

checks = [
    {"check": "same row count", "passed": len(dirty) == len(task.raw["dirty"]), "failed_rows": 0},
    {"check": "same columns", "passed": list(dirty.columns) == list(task.raw["dirty"].columns), "failed_rows": 0},
    {
        "check": "protected columns unchanged",
        "passed": dirty[protected_columns].equals(task.raw["dirty"][protected_columns]),
        "failed_rows": int(dirty[protected_columns].ne(task.raw["dirty"][protected_columns]).any(axis=1).sum()),
    },
    {
        "check": "branch_code matches order prefix",
        "passed": dirty["branch_code"].eq(expected_branch_final).all(),
        "failed_rows": int(dirty["branch_code"].ne(expected_branch_final).sum()),
    },
    {
        "check": "date canonical and parseable",
        "passed": dirty["date"].map(clean_date).eq(dirty["date"]).all(),
        "failed_rows": int(dirty["date"].map(clean_date).ne(dirty["date"]).sum()),
    },
    {
        "check": "order_type matches time",
        "passed": dirty["order_type"].eq(expected_type_final).all(),
        "failed_rows": int(dirty["order_type"].ne(expected_type_final).sum()),
    },
    {
        "check": "order_price matches menu",
        "passed": dirty["order_price"].sub(expected_price_final).abs().le(0.01).all(),
        "failed_rows": int(dirty["order_price"].sub(expected_price_final).abs().gt(0.01).sum()),
    },
    {
        "check": "coordinates have Melbourne signs",
        "passed": (dirty["customer_lat"].lt(0) & dirty["customer_lon"].gt(0)).all(),
        "failed_rows": int((~(dirty["customer_lat"].lt(0) & dirty["customer_lon"].gt(0))).sum()),
    },
    {
        "check": "distance matches road graph",
        "passed": dirty["distance_to_customer_KM"].sub(expected_distance_final).abs().le(0.001).all(),
        "failed_rows": int(dirty["distance_to_customer_KM"].sub(expected_distance_final).abs().gt(0.001).sum()),
    },
]

display(task.validate("dirty", checks))
task.step_log("dirty")

[dirty:validate] passed=9/9


,check,passed,failed_rows
0,same row count,True,0
1,same columns,True,0
2,protected columns unchanged,True,0
3,branch_code matches order prefix,True,0
4,date canonical and parseable,True,0
5,order_type matches time,True,0
6,order_price matches menu,True,0
7,coordinates have Melbourne signs,True,0
8,distance matches road graph,True,0


,dataset,step,flagged,fixed,remaining,skipped_by_tracker
0,dirty,branch_code,37,37,0,0
1,dirty,date,37,37,0,0
2,dirty,order_type,37,37,0,0
3,dirty,customer_coordinates,41,41,0,0
4,dirty,order_items,37,37,0,0
5,dirty,order_price,37,37,0,0
6,dirty,distance_to_customer_KM,37,37,0,0


## 6. Missing data

Missing data 只有 coverage/missing anomalies，所以非缺失字段可以作为干净参考。

In [48]:
missing = task.start("missing")
missing_original_na = task.raw["missing"].isna()

[missing:start] missing_cells=200


### 6.1 missing branch_code

`branch_code` 直接由 `order_id` prefix 填补。

In [49]:
missing_branch_issue = missing["branch_code"].isna()
missing_expected_branch = missing["order_id"].map(branch_from_order_id)

task.fix_values(
    dataset="missing",
    step="branch_code",
    issue_mask=missing_branch_issue,
    replacements={"branch_code": missing_expected_branch},
    sample_columns=["branch_code"],
    remaining_check=lambda df: df["branch_code"].isna(),
)

[missing:branch_code] flagged=100, fixed=100, remaining=0, skipped_by_tracker=0


,order_id_before,branch_code_before,order_id_after,branch_code_after
0,ORDX00188,NaN,ORDX00188,BK
1,ORDZ07051,NaN,ORDZ07051,NS
3,ORDC06592,NaN,ORDC06592,NS
9,ORDJ05186,NaN,ORDJ05186,TP
14,ORDA07746,NaN,ORDA07746,BK
15,ORDK01450,NaN,ORDK01450,BK
23,ORDY05307,NaN,ORDY05307,TP
27,ORDY04867,NaN,ORDY04867,TP


### 6.2 missing distance_to_customer_KM

branch 已知后，用 road graph distance 填补。

In [50]:
missing_distance_issue = missing["distance_to_customer_KM"].isna()
missing_expected_distance = missing.apply(expected_distance, axis=1)

task.fix_values(
    dataset="missing",
    step="distance_to_customer_KM",
    issue_mask=missing_distance_issue,
    replacements={"distance_to_customer_KM": missing_expected_distance},
    sample_columns=["branch_code", "customer_lat", "customer_lon", "distance_to_customer_KM"],
    remaining_check=lambda df: df["distance_to_customer_KM"].isna(),
)

[missing:distance_to_customer_KM] flagged=50, fixed=50, remaining=0, skipped_by_tracker=0


,order_id_before,branch_code_before,customer_lat_before,customer_lon_before,distance_to_customer_KM_before,order_id_after,branch_code_after,customer_lat_after,customer_lon_after,distance_to_customer_KM_after
1,ORDZ07051,NS,-37.813333,144.937581,NaN,ORDZ07051,NS,-37.813333,144.937581,10.039
9,ORDJ05186,TP,-37.811512,144.996490,NaN,ORDJ05186,TP,-37.811512,144.996490,10.612
14,ORDA07746,BK,-37.820725,144.948676,NaN,ORDA07746,BK,-37.820725,144.948676,9.214
27,ORDY04867,TP,-37.820518,144.953655,NaN,ORDY04867,TP,-37.820518,144.953655,8.665
41,ORDB09188,TP,-37.819018,144.952618,NaN,ORDB09188,TP,-37.819018,144.952618,8.594
48,ORDA06225,BK,-37.820819,144.955061,NaN,ORDA06225,BK,-37.820819,144.955061,8.591
59,ORDZ01133,NS,-37.799311,144.968292,NaN,ORDZ01133,NS,-37.799311,144.968292,6.385
64,ORDY10132,TP,-37.821631,144.983857,NaN,ORDY10132,TP,-37.821631,144.983857,9.016


### 6.3 missing delivery_fee

Delivery fee 的 business rule 已经给出：branch-specific linear model，features 是 weekend、time code、distance，并且 loyalty 有 50% discount。

In [64]:
def add_delivery_features(frame):
    result = frame.copy()
    result["date_parsed"] = pd.to_datetime(result["date"])
    result["is_weekend"] = result["date_parsed"].dt.dayofweek.isin([5, 6]).astype(int)
    result["time_code"] = result["order_type"].map(TIME_CODE).astype(int)
    return result


def adjusted_delivery_fee(frame):
    return np.where(
        frame["customerHasloyalty?"].eq(1),
        frame["delivery_fee"] * 2,
        frame["delivery_fee"],
    )


feature_columns = ["is_weekend", "time_code", "distance_to_customer_KM"]
original_fee_missing = task.raw["missing"]["delivery_fee"].isna()

model_frame = add_delivery_features(missing)
model_frame.loc[original_fee_missing, "delivery_fee"] = np.nan
model_frame["adjusted_delivery_fee"] = adjusted_delivery_fee(model_frame)
model_frame

,order_id,date,time,order_type,branch_code,order_items,order_price,customer_lat,customer_lon,customerHasloyalty?,distance_to_customer_KM,delivery_fee,date_parsed,is_weekend,time_code,adjusted_delivery_fee
0,ORDX00188,2018-07-12,14:25:21,Lunch,BK,"[('Steak', 2), ('Salad', 5)]",176.0,-37.827562,144.982878,0,8.053,13.822004,2018-07-12,0,1,13.822004
1,ORDZ07051,2018-01-07,08:10:08,Breakfast,NS,"[('Coffee', 2), ('Cereal', 7)]",162.0,-37.813333,144.937581,1,10.039,8.652942,2018-01-07,1,0,17.305884
2,ORDK05872,2018-01-27,08:50:42,Breakfast,BK,"[('Eggs', 1), ('Coffee', 8), ('Cereal', 4)]",166.0,-37.819719,144.941013,0,10.004,17.763737,2018-01-27,1,0,17.763737
3,ORDC06592,2018-05-20,12:33:48,Lunch,NS,"[('Steak', 2), ('Chicken', 7), ('Salad', 10), ...",596.0,-37.802020,144.957178,0,7.503,14.895055,2018-05-20,1,1,14.895055
4,ORDK01479,2018-10-20,17:38:01,Dinner,BK,"[('Pasta', 6), ('Shrimp', 5), ('Fish&Chips', 5...",774.0,-37.810955,144.978538,0,6.171,15.325279,2018-10-20,1,2,15.325279
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,ORDX10222,2018-11-24,18:49:00,Dinner,BK,"[('Salmon', 3), ('Fish&Chips', 1), ('Pasta', 10)]",433.0,-37.814171,144.938941,0,9.804,NaN,2018-11-24,1,2,NaN
496,ORDC05099,2018-09-12,16:27:02,Dinner,NS,"[('Fish&Chips', 3), ('Shrimp', 10)]",645.0,-37.809381,144.931753,0,9.711,16.357683,2018-09-12,0,2,16.357683
497,ORDA09719,2018-06-03,13:04:13,Lunch,BK,"[('Chicken', 1), ('Salad', 4)]",100.8,-37.816535,144.971039,0,7.091,15.351582,2018-06-03,1,1,15.351582
498,ORDK03789,2018-12-01,18:59:09,Dinner,BK,"[('Salmon', 7), ('Fish&Chips', 1), ('Pasta', 10)]",597.0,-37.817775,144.988294,0,5.863,15.346807,2018-12-01,1,2,15.346807


In [52]:
fee_models = {}
fee_metrics = []
for branch_code, branch_rows in model_frame.loc[model_frame["delivery_fee"].notna()].groupby("branch_code"):
    model = LinearRegression()
    model.fit(branch_rows[feature_columns], branch_rows["adjusted_delivery_fee"])
    fee_models[branch_code] = model
    fee_metrics.append(
        {
            "branch_code": branch_code,
            "training_rows": len(branch_rows),
            "r2_score": model.score(branch_rows[feature_columns], branch_rows["adjusted_delivery_fee"]),
            "coef_is_weekend": model.coef_[0],
            "coef_time_code": model.coef_[1],
            "coef_distance": model.coef_[2],
        }
    )

fee_metrics = pd.DataFrame(fee_metrics)
fee_metrics

,branch_code,training_rows,r2_score,coef_is_weekend,coef_time_code,coef_distance
0,BK,153,0.994620,2.525824,0.958615,1.057095
1,NS,128,0.967024,1.929362,0.537725,1.016335
2,TP,169,0.951958,1.481821,0.735954,0.850610


In [53]:
prediction_frame = add_delivery_features(missing.loc[original_fee_missing])
predicted_fees = []

for _, row in prediction_frame.iterrows():
    feature_row = pd.DataFrame([{column: row[column] for column in feature_columns}])
    predicted_fee = float(fee_models[row["branch_code"]].predict(feature_row)[0])
    if int(row["customerHasloyalty?"]) == 1:
        predicted_fee = predicted_fee / 2
    predicted_fees.append(round(predicted_fee, 6))

predicted_fees = pd.Series(predicted_fees, index=prediction_frame.index)

task.fix_values(
    dataset="missing",
    step="delivery_fee",
    issue_mask=original_fee_missing,
    replacements={"delivery_fee": predicted_fees},
    sample_columns=["branch_code", "date", "order_type", "customerHasloyalty?", "distance_to_customer_KM", "delivery_fee"],
    remaining_check=lambda df: df["delivery_fee"].isna(),
)

[missing:delivery_fee] flagged=50, fixed=50, remaining=0, skipped_by_tracker=0


,order_id_before,branch_code_before,date_before,order_type_before,customerHasloyalty?_before,distance_to_customer_KM_before,delivery_fee_before,order_id_after,branch_code_after,date_after,order_type_after,customerHasloyalty?_after,distance_to_customer_KM_after,delivery_fee_after
7,ORDC10700,NS,2018-06-28,Lunch,0,9.658,NaN,ORDC10700,NS,2018-06-28,Lunch,0,9.658,15.194097
33,ORDJ08835,TP,2018-06-14,Lunch,0,8.517,NaN,ORDJ08835,TP,2018-06-14,Lunch,0,8.517,12.017467
38,ORDJ07186,TP,2018-10-29,Dinner,0,11.443,NaN,ORDJ07186,TP,2018-10-29,Dinner,0,11.443,15.242306
42,ORDC03961,NS,2018-01-11,Breakfast,1,8.147,NaN,ORDC03961,NS,2018-01-11,Breakfast,1,8.147,6.560345
61,ORDJ09090,TP,2018-07-21,Lunch,0,8.136,NaN,ORDJ09090,TP,2018-07-21,Lunch,0,8.136,13.175206
62,ORDI02574,NS,2018-06-04,Lunch,0,8.058,NaN,ORDI02574,NS,2018-06-04,Lunch,0,8.058,13.567961
66,ORDJ04126,TP,2018-03-20,Breakfast,0,8.575,NaN,ORDJ04126,TP,2018-03-20,Breakfast,0,8.575,11.330848
83,ORDZ01109,NS,2018-08-15,Lunch,0,7.337,NaN,ORDZ01109,NS,2018-08-15,Lunch,0,7.337,12.835184


In [54]:
missing_fee_audit = prediction_frame[
    ["order_id", "branch_code", "date", "order_type", "customerHasloyalty?", "distance_to_customer_KM"]
].copy()
missing_fee_audit["imputed_delivery_fee"] = predicted_fees
missing_fee_audit.head(8)

,order_id,branch_code,date,order_type,customerHasloyalty?,distance_to_customer_KM,imputed_delivery_fee
7,ORDC10700,NS,2018-06-28,Lunch,0,9.658,15.194097
33,ORDJ08835,TP,2018-06-14,Lunch,0,8.517,12.017467
38,ORDJ07186,TP,2018-10-29,Dinner,0,11.443,15.242306
42,ORDC03961,NS,2018-01-11,Breakfast,1,8.147,6.560345
61,ORDJ09090,TP,2018-07-21,Lunch,0,8.136,13.175206
62,ORDI02574,NS,2018-06-04,Lunch,0,8.058,13.567961
66,ORDJ04126,TP,2018-03-20,Breakfast,0,8.575,11.330848
83,ORDZ01109,NS,2018-08-15,Lunch,0,7.337,12.835184


### 6.4 missing validation

In [55]:
missing_expected_branch_final = missing["order_id"].map(branch_from_order_id)
missing_expected_distance_final = missing.apply(expected_distance, axis=1)
min_fee_r2 = float(fee_metrics["r2_score"].min())

checks = [
    {"check": "same row count", "passed": len(missing) == len(task.raw["missing"]), "failed_rows": 0},
    {"check": "same columns", "passed": list(missing.columns) == list(task.raw["missing"].columns), "failed_rows": 0},
    {"check": "no missing values", "passed": not missing.isna().any().any(), "failed_rows": int(missing.isna().any(axis=1).sum())},
    {
        "check": "branch_code matches order prefix",
        "passed": missing["branch_code"].eq(missing_expected_branch_final).all(),
        "failed_rows": int(missing["branch_code"].ne(missing_expected_branch_final).sum()),
    },
    {
        "check": "distance matches road graph",
        "passed": missing["distance_to_customer_KM"].sub(missing_expected_distance_final).abs().le(0.001).all(),
        "failed_rows": int(missing["distance_to_customer_KM"].sub(missing_expected_distance_final).abs().gt(0.001).sum()),
    },
    {"check": "delivery_fee model min R2 >= 0.95", "passed": min_fee_r2 >= 0.95, "failed_rows": 0 if min_fee_r2 >= 0.95 else 1},
    {"check": "delivery_fee positive", "passed": missing["delivery_fee"].gt(0).all(), "failed_rows": int(missing["delivery_fee"].le(0).sum())},
]

display(task.validate("missing", checks))
task.step_log("missing")

[missing:validate] passed=7/7


,check,passed,failed_rows
0,same row count,True,0
1,same columns,True,0
2,no missing values,True,0
3,branch_code matches order prefix,True,0
4,distance matches road graph,True,0
5,delivery_fee model min R2 >= 0.95,True,0
6,delivery_fee positive,True,0


,dataset,step,flagged,fixed,remaining,skipped_by_tracker
0,missing,branch_code,100,100,0,0
1,missing,distance_to_customer_KM,50,50,0,0
2,missing,delivery_fee,50,50,0,0


## 7. Outlier data

Outlier data 不修值，只删除 delivery fee outlier rows。

In [56]:
outlier = task.start("outlier")

[outlier:start] target attribute: delivery_fee


### 7.1 Fit delivery fee model and compute residuals

这里不再封装大函数。直接按 branch 循环：fit model、predict、算 residual。课堂上重点看三件事：

1. loyalty fee 先还原成 adjusted fee；
2. 每个 branch 单独 fit linear regression；
3. outlier 用 absolute residual 判断。

In [57]:
outlier_model_frame = add_delivery_features(outlier)
outlier_model_frame["adjusted_delivery_fee"] = adjusted_delivery_fee(outlier_model_frame)

outlier_residual_frames = []
outlier_metric_rows = []

for branch_code, branch_rows in outlier_model_frame.groupby("branch_code"):
    model = LinearRegression()
    model.fit(branch_rows[feature_columns], branch_rows["adjusted_delivery_fee"])

    predicted = model.predict(branch_rows[feature_columns])
    residual = branch_rows["adjusted_delivery_fee"] - predicted

    branch_residuals = branch_rows[
        ["order_id", "branch_code", "delivery_fee", "customerHasloyalty?", "is_weekend", "time_code", "distance_to_customer_KM"]
    ].copy()
    branch_residuals["adjusted_delivery_fee"] = branch_rows["adjusted_delivery_fee"]
    branch_residuals["predicted_adjusted_fee"] = predicted
    branch_residuals["abs_residual"] = residual.abs()
    outlier_residual_frames.append(branch_residuals)

    outlier_metric_rows.append({
        "branch_code": branch_code,
        "rows": len(branch_rows),
        "r2_score": model.score(branch_rows[feature_columns], branch_rows["adjusted_delivery_fee"]),
        "mean_abs_residual": residual.abs().mean(),
    })

outlier_residuals = pd.concat(outlier_residual_frames)
initial_outlier_metrics = pd.DataFrame(outlier_metric_rows)
initial_outlier_metrics

,branch_code,rows,r2_score,mean_abs_residual
0,BK,184,0.745726,0.492381
1,NS,158,0.413726,1.071325
2,TP,158,0.234724,1.131314


### 7.2 Branch-wise IQR rule

因为 fee model 是 branch-specific，outlier threshold 也按 branch 分开算。

In [58]:
flagged_frames = []
bounds = []

for branch_code, branch_rows in outlier_residuals.groupby("branch_code"):
    q1, q3 = branch_rows["abs_residual"].quantile([0.25, 0.75])
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    branch_flagged = branch_rows.copy()
    branch_flagged["outlier_bound"] = upper_bound
    branch_flagged["is_delivery_fee_outlier"] = branch_flagged["abs_residual"].gt(upper_bound)
    flagged_frames.append(branch_flagged)

    bounds.append(
        {
            "branch_code": branch_code,
            "q1_abs_residual": q1,
            "q3_abs_residual": q3,
            "upper_bound": upper_bound,
            "outlier_count": int(branch_flagged["is_delivery_fee_outlier"].sum()),
        }
    )

outlier_bounds = pd.DataFrame(bounds)
flagged_residuals = pd.concat(flagged_frames)
delivery_fee_outliers = flagged_residuals.loc[flagged_residuals["is_delivery_fee_outlier"]].sort_values(
    ["branch_code", "abs_residual"],
    ascending=[True, False],
)

display(outlier_bounds)
delivery_fee_outliers.head(10)

,branch_code,q1_abs_residual,q3_abs_residual,upper_bound,outlier_count
0,BK,0.095200,0.412606,0.888715,9
1,NS,0.163942,0.909859,2.028733,14
2,TP,0.182145,0.680127,1.427099,21


,order_id,branch_code,delivery_fee,customerHasloyalty?,is_weekend,time_code,distance_to_customer_KM,adjusted_delivery_fee,predicted_adjusted_fee,abs_residual,outlier_bound,is_delivery_fee_outlier
199,ORDX02124,BK,12.910252,1,1,0,9.183,25.820504,16.719327,9.101177,0.888715,True
318,ORDA03445,BK,7.511299,0,0,1,8.861,7.511299,14.771996,7.260696,0.888715,True
442,ORDK00636,BK,22.373108,0,0,2,8.356,22.373108,15.346893,7.026215,0.888715,True
91,ORDK09590,BK,6.653379,0,0,0,8.358,6.653379,13.128992,6.475613,0.888715,True
93,ORDK00865,BK,19.871249,0,0,1,7.741,19.871249,13.585210,6.286039,0.888715,True
490,ORDK02256,BK,5.216878,0,0,0,5.604,5.216878,10.210771,4.993893,0.888715,True
387,ORDX02830,BK,14.314189,0,0,2,3.454,14.314189,10.152586,4.161602,0.888715,True
486,ORDX08202,BK,19.417588,0,1,2,10.494,19.417588,20.328522,0.910934,0.888715,True
496,ORDA04575,BK,6.875172,1,0,0,8.094,13.750345,12.849250,0.901095,0.888715,True
397,ORDZ10737,NS,27.790624,0,1,2,10.622,27.790624,19.543467,8.247158,2.028733,True


In [59]:
outlier_issue = outlier["order_id"].isin(delivery_fee_outliers["order_id"])

removed_outliers = task.remove_rows(
    dataset="outlier",
    step="delivery_fee",
    issue_mask=outlier_issue,
    sample_columns=["branch_code", "delivery_fee", "customerHasloyalty?", "distance_to_customer_KM"],
)

outlier = task.frame("outlier")
removed_outliers

[outlier:delivery_fee] flagged=44, fixed=44, remaining=0, skipped_by_tracker=0
[outlier:delivery_fee] rows_before=500, rows_after=456


,order_id,branch_code,delivery_fee,customerHasloyalty?,distance_to_customer_KM
8,ORDZ01820,NS,27.096514,0,10.006
10,ORDJ06997,TP,6.921187,0,11.177
27,ORDY04830,TP,6.373714,0,8.601
29,ORDZ09868,NS,22.611795,0,9.588
54,ORDI00907,NS,21.985400,0,7.111
57,ORDC01694,NS,22.234221,0,6.669
63,ORDB04124,TP,6.483729,0,10.199
71,ORDZ01558,NS,12.924880,1,9.666
91,ORDK09590,BK,6.653379,0,8.358
92,ORDB08727,TP,6.754654,0,7.957


### 7.3 outlier validation

删除后，用同样的短 loop 重新训练 retained rows，确认正常数据更符合 delivery fee model。

In [60]:
retained_model_frame = add_delivery_features(outlier)
retained_model_frame["adjusted_delivery_fee"] = adjusted_delivery_fee(retained_model_frame)

retained_metric_rows = []
for branch_code, branch_rows in retained_model_frame.groupby("branch_code"):
    model = LinearRegression()
    model.fit(branch_rows[feature_columns], branch_rows["adjusted_delivery_fee"])
    predicted = model.predict(branch_rows[feature_columns])
    residual = branch_rows["adjusted_delivery_fee"] - predicted

    retained_metric_rows.append({
        "branch_code": branch_code,
        "retained_rows": len(branch_rows),
        "r2_score": model.score(branch_rows[feature_columns], branch_rows["adjusted_delivery_fee"]),
        "mean_abs_residual": residual.abs().mean(),
    })

retained_metrics = pd.DataFrame(retained_metric_rows)
min_retained_r2 = float(retained_metrics["r2_score"].min())

checks = [
    {"check": "same columns", "passed": list(outlier.columns) == list(task.raw["outlier"].columns), "failed_rows": 0},
    {"check": "rows removed only", "passed": len(outlier) <= len(task.raw["outlier"]), "failed_rows": 0},
    {"check": "delivery_fee retained model min R2 >= 0.95", "passed": min_retained_r2 >= 0.95, "failed_rows": 0 if min_retained_r2 >= 0.95 else 1},
    {"check": "delivery_fee positive", "passed": outlier["delivery_fee"].gt(0).all(), "failed_rows": int(outlier["delivery_fee"].le(0).sum())},
]

display(task.validate("outlier", checks))
retained_metrics

[outlier:validate] passed=4/4


,check,passed,failed_rows
0,same columns,True,0
1,rows removed only,True,0
2,delivery_fee retained model min R2 >= 0.95,True,0
3,delivery_fee positive,True,0


,branch_code,retained_rows,r2_score,mean_abs_residual
0,BK,175,0.982534,0.227072
1,NS,144,0.960104,0.251420
2,TP,137,0.961993,0.222849


## 8. Export and final QA

最后导出三个 CSV。这里 helper 只负责统一输出文件名和 read-back QA。

In [61]:
task.compare_with_saved_outputs()

[compare] current in-memory results checked against saved CSV files


,dataset,same_shape,same_columns,numeric_close
0,dirty,True,True,True
1,missing,True,True,True
2,outlier,True,True,True


In [62]:
task.export_all()

[export] Group024_dirty_data_solution.csv: rows=500, columns=12
[export] Group024_missing_data_solution.csv: rows=500, columns=12
[export] Group024_outlier_data_solution.csv: rows=456, columns=12


In [ ]:
display(task.read_back_outputs())
task.step_log()

[read-back] exported CSV files parsed successfully


,file,rows,columns,missing_cells,same_columns
0,Group024_dirty_data_solution.csv,500,12,0,True
1,Group024_missing_data_solution.csv,500,12,0,True
2,Group024_outlier_data_solution.csv,456,12,0,True


,dataset,step,flagged,fixed,remaining,skipped_by_tracker
0,dirty,branch_code,37,37,0,0
1,dirty,date,37,37,0,0
2,dirty,order_type,37,37,0,0
3,dirty,customer_coordinates,41,41,0,0
4,dirty,order_items,37,37,0,0
5,dirty,order_price,37,37,0,0
6,dirty,distance_to_customer_KM,37,37,0,0
7,missing,branch_code,100,100,0,0
8,missing,distance_to_customer_KM,50,50,0,0
9,missing,delivery_fee,50,50,0,0


## Teaching summary

这个版本更适合课堂讲解：

- 规则从 assignment PDF 来；
- 每个 attribute 的判断条件在 notebook 里看得到；
- helper 只是统一执行和记录；
- final validation 只收口检查，不临时修数据。